In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
import torch

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'
models_path = project_path + '/models'
results_path = project_path + '/results'

channel = "A-8"

train_data = np.load(
    train_path + "/" + channel + ".npy"
)

test_data = np.load(
    test_path + "/" + channel + ".npy"
)

train_telemetry = train_data[:, 0]
test_telemetry = test_data[:, 0]

window_size = 50

print("Project path:", project_path)
print("Channel:", channel)
print("Train telemetry shape:", train_telemetry.shape)
print("Test telemetry shape:", test_telemetry.shape)
print("Window size:", window_size)

Mounted at /content/drive
Project path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection
Channel: A-8
Train telemetry shape: (762,)
Test telemetry shape: (8375,)
Window size: 50


##Create Time-Series Windows

We'll create the same windows as SERIAL 22/23. This keeps the experiment comparable.

In [2]:
def create_windows(telemetry, window_size=50):
    windows = []

    for i in range(len(telemetry) - window_size + 1):
        windows.append(
            telemetry[i:i + window_size]
        )

    return np.array(windows)


train_windows = create_windows(
    train_telemetry,
    window_size
)

test_windows = create_windows(
    test_telemetry,
    window_size
)

X_train = torch.tensor(
    train_windows,
    dtype=torch.float32
).unsqueeze(-1)

X_test = torch.tensor(
    test_windows,
    dtype=torch.float32
).unsqueeze(-1)

print("Training windows:", X_train.shape)
print("Test windows:", X_test.shape)

Training windows: torch.Size([713, 50, 1])
Test windows: torch.Size([8326, 50, 1])


#Build the Lightweight LSTM Autoencoder

In [3]:
class LightweightLSTMAutoencoder(torch.nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=16,
        latent_size=16
    ):
        super().__init__()

        self.encoder = torch.nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.encoder_fc = torch.nn.Linear(
            hidden_size,
            latent_size
        )

        self.decoder_fc = torch.nn.Linear(
            latent_size,
            hidden_size
        )

        self.decoder = torch.nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.output_layer = torch.nn.Linear(
            hidden_size,
            input_size
        )

    def forward(self, x):
        encoded_sequence, (hidden, cell) = self.encoder(x)

        latent = self.encoder_fc(
            hidden[-1]
        )

        decoder_input = self.decoder_fc(
            latent
        )

        decoder_input = decoder_input.unsqueeze(1)

        decoder_input = decoder_input.repeat(
            1,
            x.size(1),
            1
        )

        decoded_sequence, _ = self.decoder(
            decoder_input
        )

        output = self.output_layer(
            decoded_sequence
        )

        return output


lightweight_model = LightweightLSTMAutoencoder()

total_parameters = sum(
    parameter.numel()
    for parameter in lightweight_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in lightweight_model.parameters()
    if parameter.requires_grad
)

print("Hidden size:", 16)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Hidden size: 16
Total parameters: 3953
Trainable parameters: 3953


###Parameter reduction

| Model                        | Hidden size | Parameters |
| ---------------------------- | ----------: | ---------: |
| Original LSTM Autoencoder    |          32 |     14,033 |
| Lightweight LSTM Autoencoder |          16 |  **3,953** |

That's a reduction of about 71.8% in parameters.

##Train the Lightweight Model

In [4]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    lightweight_model.parameters(),
    lr=0.001
)

epochs = 50

start_time = time.perf_counter()

for epoch in range(epochs):
    optimizer.zero_grad()

    reconstructed = lightweight_model(X_train)

    loss = criterion(
        reconstructed,
        X_train
    )

    loss.backward()
    optimizer.step()

end_time = time.perf_counter()

lightweight_training_time = end_time - start_time
lightweight_average_epoch_time = (
    lightweight_training_time / epochs
)

print("Training epochs:", epochs)
print(
    "Total training time (seconds):",
    round(lightweight_training_time, 6)
)
print(
    "Average time per epoch (seconds):",
    round(lightweight_average_epoch_time, 6)
)
print(
    "Final training loss:",
    round(loss.item(), 6)
)

Training epochs: 50
Total training time (seconds): 8.069645
Average time per epoch (seconds): 0.161393
Final training loss: 0.316677


###We now have the first computational comparison:

| Metric              | Original LSTM (32) | Lightweight LSTM (16) |
| ------------------- | -----------------: | --------------------: |
| Parameters          |             14,033 |             **3,953** |
| Training time       |            13.16 s |            **8.07 s** |
| Avg. epoch          |           0.2632 s |          **0.1614 s** |
| Final training loss |            0.08049 |               0.31668 |



###Measure Inference Time

Now we test how fast the lightweight model processes the 8,326 test windows.

In [5]:
lightweight_model.eval()

with torch.no_grad():
    start_time = time.perf_counter()

    lightweight_reconstructed = lightweight_model(X_test)

    end_time = time.perf_counter()

lightweight_inference_time = end_time - start_time
lightweight_time_per_window = (
    lightweight_inference_time / len(X_test)
)

print("Test windows:", len(X_test))
print(
    "Total inference time (seconds):",
    round(lightweight_inference_time, 6)
)
print(
    "Inference time per window (ms):",
    round(lightweight_time_per_window * 1000, 6)
)

Test windows: 8326
Total inference time (seconds): 0.848604
Inference time per window (ms): 0.101922


##This is a strong computational result.

| Metric           | Original LSTM | Lightweight LSTM |
| ---------------- | ------------: | ---------------: |
| Hidden size      |            32 |               16 |
| Parameters       |        14,033 |        **3,953** |
| Training time    |       13.16 s |       **8.07 s** |
| Inference time   |        2.61 s |       **0.85 s** |
| Inference/window |      0.313 ms |     **0.102 ms** |


The lightweight model has about 71.8% fewer parameters and its measured inference time is substantially lower in this run.

##Calculate Reconstruction Errors

Now we'll calculate the reconstruction error for both training and test windows and create the anomaly threshold using only training data.

In [6]:
with torch.no_grad():
    train_reconstructed = lightweight_model(X_train)

    train_errors = torch.mean(
        (train_reconstructed - X_train) ** 2,
        dim=(1, 2)
    ).numpy()

    test_errors = torch.mean(
        (lightweight_reconstructed - X_test) ** 2,
        dim=(1, 2)
    ).numpy()

threshold = np.percentile(
    train_errors,
    99
)

print("Training error mean:", train_errors.mean())
print("Training error max:", train_errors.max())
print("Test error mean:", test_errors.mean())
print("Test error max:", test_errors.max())
print("99th percentile threshold:", threshold)

Training error mean: 0.30051255
Training error max: 2.7802835
Test error mean: 0.04221974
Test error max: 0.47169867
99th percentile threshold: 2.6175277


Here we Notice that the test errors are much lower than the training errors. This means the 99th-percentile threshold is very high, so the lightweight model may classify few or no test windows as anomalies.

That's not something we should “fix” just to get better numbers. It is an experimental finding.

##Generate Predictions

Now let's see how many test windows the lightweight model actually flags

In [7]:
lightweight_predictions = (
    test_errors > threshold
).astype(int)

print(
    "Normal windows:",
    np.sum(lightweight_predictions == 0)
)

print(
    "Anomaly windows:",
    np.sum(lightweight_predictions == 1)
)

print(
    "Total windows:",
    len(lightweight_predictions)
)

Normal windows: 8326
Anomaly windows: 0
Total windows: 8326


##Evaluate the Lightweight Model

In [8]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

ground_truth = np.zeros(
    len(test_telemetry),
    dtype=int
)

ground_truth[4569:8375] = 1

ground_truth_windows = ground_truth[
    window_size - 1:
]

precision = precision_score(
    ground_truth_windows,
    lightweight_predictions,
    zero_division=0
)

recall = recall_score(
    ground_truth_windows,
    lightweight_predictions,
    zero_division=0
)

f1 = f1_score(
    ground_truth_windows,
    lightweight_predictions,
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    ground_truth_windows,
    lightweight_predictions
).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("Precision:", precision)
print("Recall:", recall)
print("F1 score:", f1)
print("False Positive Rate:", fpr)
print("False Negative Rate:", fnr)

print("\nConfusion Matrix:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Precision: 0.0
Recall: 0.0
F1 score: 0.0
False Positive Rate: 0.0
False Negative Rate: 1.0

Confusion Matrix:
TN: 4520
FP: 0
FN: 3806
TP: 0


##Compare Original vs Lightweight

Now we'll put the results together without changing either model.

In [9]:
comparison = pd.DataFrame([
    {
        "model": "Original LSTM Autoencoder",
        "hidden_size": 32,
        "parameters": 14033,
        "training_time_seconds": 13.160519,
        "inference_time_seconds": 2.609967,
        "precision": 0.2775065,
        "recall": 0.4479769,
        "f1_score": 0.3427136,
        "false_positive_rate": 0.9820796,
        "false_negative_rate": 0.5520231
    },
    {
        "model": "Lightweight LSTM Autoencoder",
        "hidden_size": 16,
        "parameters": 3953,
        "training_time_seconds": lightweight_training_time,
        "inference_time_seconds": lightweight_inference_time,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "false_negative_rate": fnr
    }
])

comparison

,model,hidden_size,parameters,training_time_seconds,inference_time_seconds,precision,recall,f1_score,false_positive_rate,false_negative_rate
0,Original LSTM Autoencoder,32,14033,13.160519,2.609967,0.277506,0.447977,0.342714,0.98208,0.552023
1,Lightweight LSTM Autoencoder,16,3953,8.069645,0.848604,0.000000,0.000000,0.000000,0.00000,1.000000


The smaller model is much cheaper computationally, but it lost anomaly-detection capability under the current training/threshold setup.

This gives us a real trade-off:

Computational efficiency improved substantially, but detection performance degraded severely.

##Save the Optimization Results

In [10]:
lightweight_results_path = (
    results_path + "/lightweight_optimization_A8.csv"
)

comparison.to_csv(
    lightweight_results_path,
    index=False
)

print(
    "Saved:",
    lightweight_results_path
)

Saved: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/lightweight_optimization_A8.csv


##Now We'll create the hidden-size-24 model, keeping everything else unchanged

In [11]:
class MediumLSTMAutoencoder(torch.nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=24,
        latent_size=16
    ):
        super().__init__()

        self.encoder = torch.nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.encoder_fc = torch.nn.Linear(
            hidden_size,
            latent_size
        )

        self.decoder_fc = torch.nn.Linear(
            latent_size,
            hidden_size
        )

        self.decoder = torch.nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.output_layer = torch.nn.Linear(
            hidden_size,
            input_size
        )

    def forward(self, x):
        encoded_sequence, (hidden, cell) = self.encoder(x)

        latent = self.encoder_fc(
            hidden[-1]
        )

        decoder_input = self.decoder_fc(
            latent
        )

        decoder_input = decoder_input.unsqueeze(1)

        decoder_input = decoder_input.repeat(
            1,
            x.size(1),
            1
        )

        decoded_sequence, _ = self.decoder(
            decoder_input
        )

        output = self.output_layer(
            decoded_sequence
        )

        return output


medium_model = MediumLSTMAutoencoder()

total_parameters = sum(
    parameter.numel()
    for parameter in medium_model.parameters()
)

print("Hidden size:", 24)
print("Total parameters:", total_parameters)

Hidden size: 24
Total parameters: 8225


## Now we have three model sizes:

| Model       | Hidden size | Parameters |
| ----------- | ----------: | ---------: |
| Original    |          32 |     14,033 |
| Medium      |          24 |  **8,225** |
| Lightweight |          16 |      3,953 |



#Train the 24-unit Model

In [12]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    medium_model.parameters(),
    lr=0.001
)

epochs = 50

start_time = time.perf_counter()

for epoch in range(epochs):
    optimizer.zero_grad()

    reconstructed = medium_model(X_train)

    loss = criterion(
        reconstructed,
        X_train
    )

    loss.backward()
    optimizer.step()

end_time = time.perf_counter()

medium_training_time = end_time - start_time
medium_average_epoch_time = (
    medium_training_time / epochs
)

print("Training epochs:", epochs)
print(
    "Total training time (seconds):",
    round(medium_training_time, 6)
)
print(
    "Average time per epoch (seconds):",
    round(medium_average_epoch_time, 6)
)
print(
    "Final training loss:",
    round(loss.item(), 6)
)

Training epochs: 50
Total training time (seconds): 25.123471
Average time per epoch (seconds): 0.502469
Final training loss: 0.152024


the 24-unit model has lower final training loss (0.152) than the 16-unit model (0.317), but its measured training time was actually higher than the original run. That timing can vary in Colab because of CPU/runtime conditions, so we shouldn't interpret one timing run too strongly.

#Measure 24-unit Inference Time

In [13]:
medium_model.eval()

with torch.no_grad():
    start_time = time.perf_counter()

    medium_reconstructed = medium_model(X_test)

    end_time = time.perf_counter()

medium_inference_time = end_time - start_time

medium_time_per_window = (
    medium_inference_time / len(X_test)
)

print("Test windows:", len(X_test))
print(
    "Total inference time (seconds):",
    round(medium_inference_time, 6)
)
print(
    "Inference time per window (ms):",
    round(medium_time_per_window * 1000, 6)
)

Test windows: 8326
Total inference time (seconds): 1.491779
Inference time per window (ms): 0.179171


####Now we need to see whether the 24-unit model can actually detect anomalies.

###Calculate Reconstruction Errors

In [14]:
with torch.no_grad():
    medium_train_reconstructed = medium_model(X_train)

    medium_train_errors = torch.mean(
        (medium_train_reconstructed - X_train) ** 2,
        dim=(1, 2)
    ).numpy()

    medium_test_errors = torch.mean(
        (medium_reconstructed - X_test) ** 2,
        dim=(1, 2)
    ).numpy()

medium_threshold = np.percentile(
    medium_train_errors,
    99
)

print(
    "Training error mean:",
    medium_train_errors.mean()
)

print(
    "Training error max:",
    medium_train_errors.max()
)

print(
    "Test error mean:",
    medium_test_errors.mean()
)

print(
    "Test error max:",
    medium_test_errors.max()
)

print(
    "99th percentile threshold:",
    medium_threshold
)

Training error mean: 0.1434668
Training error max: 2.3055472
Test error mean: 0.026397185
Test error max: 0.23948596
99th percentile threshold: 2.1777232


So the threshold is far above every test reconstruction error. This suggests the 24-unit model will probably flag zero test windows, just like the 16-unit model.

##Generate Predictions

In [15]:
medium_predictions = (
    medium_test_errors > medium_threshold
).astype(int)

print(
    "Normal windows:",
    np.sum(medium_predictions == 0)
)

print(
    "Anomaly windows:",
    np.sum(medium_predictions == 1)
)

print(
    "Total windows:",
    len(medium_predictions)
)

Normal windows: 8326
Anomaly windows: 0
Total windows: 8326


###Evaluate the 24-unit Model

In [16]:
medium_precision = precision_score(
    ground_truth_windows,
    medium_predictions,
    zero_division=0
)

medium_recall = recall_score(
    ground_truth_windows,
    medium_predictions,
    zero_division=0
)

medium_f1 = f1_score(
    ground_truth_windows,
    medium_predictions,
    zero_division=0
)

medium_tn, medium_fp, medium_fn, medium_tp = (
    confusion_matrix(
        ground_truth_windows,
        medium_predictions
    ).ravel()
)

medium_fpr = medium_fp / (
    medium_fp + medium_tn
)

medium_fnr = medium_fn / (
    medium_fn + medium_tp
)

print("Precision:", medium_precision)
print("Recall:", medium_recall)
print("F1 score:", medium_f1)
print("False Positive Rate:", medium_fpr)
print("False Negative Rate:", medium_fnr)

print("\nConfusion Matrix:")
print("TN:", medium_tn)
print("FP:", medium_fp)
print("FN:", medium_fn)
print("TP:", medium_tp)

Precision: 0.0
Recall: 0.0
F1 score: 0.0
False Positive Rate: 0.0
False Negative Rate: 1.0

Confusion Matrix:
TN: 4520
FP: 0
FN: 3806
TP: 0


##A-8 lightweight optimization:

| Model            | Hidden | Parameters |   Inference |        F1 | Recall |   FPR |
| ---------------- | -----: | ---------: | ----------: | --------: | -----: | ----: |
| Original LSTM    |     32 |     14,033 |     2.610 s |     0.343 |  0.448 | 0.982 |
| Medium LSTM      |     24 |      8,225 |     1.492 s | **0.000** |  0.000 | 0.000 |
| Lightweight LSTM |     16 |      3,953 | **0.849 s** | **0.000** |  0.000 | 0.000 |

Both smaller models have substantially lower computational cost, but neither detected an A-8 anomaly using the same thresholding procedure.

Reducing LSTM hidden size can significantly reduce computational requirements, but aggressive parameter reduction may compromise anomaly-detection capability.

#Save the 32/24/16 Comparison

In [17]:
hidden_size_comparison = pd.DataFrame([
    {
        "model": "Original LSTM Autoencoder",
        "hidden_size": 32,
        "parameters": 14033,
        "training_time_seconds": 13.160519,
        "inference_time_seconds": 2.609967,
        "precision": 0.2775065,
        "recall": 0.4479769,
        "f1_score": 0.3427136,
        "false_positive_rate": 0.9820796,
        "false_negative_rate": 0.5520231
    },
    {
        "model": "Medium LSTM Autoencoder",
        "hidden_size": 24,
        "parameters": 8225,
        "training_time_seconds": medium_training_time,
        "inference_time_seconds": medium_inference_time,
        "precision": medium_precision,
        "recall": medium_recall,
        "f1_score": medium_f1,
        "false_positive_rate": medium_fpr,
        "false_negative_rate": medium_fnr
    },
    {
        "model": "Lightweight LSTM Autoencoder",
        "hidden_size": 16,
        "parameters": 3953,
        "training_time_seconds": lightweight_training_time,
        "inference_time_seconds": lightweight_inference_time,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "false_negative_rate": fnr
    }
])

hidden_size_results_path = (
    results_path + "/lstm_hidden_size_comparison_A8.csv"
)

hidden_size_comparison.to_csv(
    hidden_size_results_path,
    index=False
)

print(hidden_size_comparison)
print("\nSaved:", hidden_size_results_path)

                          model  hidden_size  parameters  \
0     Original LSTM Autoencoder           32       14033   
1       Medium LSTM Autoencoder           24        8225   
2  Lightweight LSTM Autoencoder           16        3953   

   training_time_seconds  inference_time_seconds  precision    recall  \
0              13.160519                2.609967   0.277506  0.447977   
1              25.123471                1.491779   0.000000  0.000000   
2               8.069645                0.848604   0.000000  0.000000   

   f1_score  false_positive_rate  false_negative_rate  
0  0.342714              0.98208             0.552023  
1  0.000000              0.00000             1.000000  
2  0.000000              0.00000             1.000000  

Saved: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/lstm_hidden_size_comparison_A8.csv


Reducing the LSTM hidden size from 32 to 24 and 16 reduced the number of trainable parameters from 14,033 to 8,225 and 3,953, respectively. Inference time also decreased from 2.61 s to 1.49 s and 0.85 s for 8,326 test windows. However, both reduced configurations produced zero detected anomalies on A-8 when the threshold was determined from the 99th percentile of training reconstruction error, resulting in an F1 score of 0. These results indicate a clear computational-efficiency versus detection-performance trade-off for hidden-size reduction.

##github commit

In [18]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status --short

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (52/52), done.
 M notebooks/13_evaluate_global_zscore.ipynb
 M notebooks/14_rolling_zscore.ipynb
 M notebooks/15_compare_statistical_methods.ipynb
 M notebooks/16_isolation_forest.ipynb
 M notebooks/17_evaluate_isolation_forest.ipynb
 M notebooks/18_one_class_svm.ipynb
 M notebooks/19_compare_classical_methods.ipynb
 M notebooks/20_autoencoder.ipynb
 M notebooks/21_analyze_autoencoder.ipynb
 M notebooks/22_time_series_windows.ipynb
 M notebooks/23_lstm_autoencoder.ipynb
 M notebooks/24_evaluate_lstm_autoencoder.ipynb
 M notebooks/25_model_comparison_and_research_gap.ipynb
 M notebooks/26_explainable_ai.ipynb
 M notebooks/27_computational_cost_analysis.ipynb
?? notebooks/28_lightweight_optimization.ipynb
?? results/lightweight_optimization_A8.csv
?? results/lstm_hidden_size_comparison_A8.csv


In [19]:
!git add notebooks/28_lightweight_optimization.ipynb
!git add results/lightweight_optimization_A8.csv
!git add results/lstm_hidden_size_comparison_A8.csv

In [20]:
!git status --short

 M notebooks/13_evaluate_global_zscore.ipynb
 M notebooks/14_rolling_zscore.ipynb
 M notebooks/15_compare_statistical_methods.ipynb
 M notebooks/16_isolation_forest.ipynb
 M notebooks/17_evaluate_isolation_forest.ipynb
 M notebooks/18_one_class_svm.ipynb
 M notebooks/19_compare_classical_methods.ipynb
 M notebooks/20_autoencoder.ipynb
 M notebooks/21_analyze_autoencoder.ipynb
 M notebooks/22_time_series_windows.ipynb
 M notebooks/23_lstm_autoencoder.ipynb
 M notebooks/24_evaluate_lstm_autoencoder.ipynb
 M notebooks/25_model_comparison_and_research_gap.ipynb
 M notebooks/26_explainable_ai.ipynb
 M notebooks/27_computational_cost_analysis.ipynb
A  notebooks/28_lightweight_optimization.ipynb
A  results/lightweight_optimization_A8.csv
A  results/lstm_hidden_size_comparison_A8.csv


In [21]:
!git commit -m "Complete SERIAL 28 lightweight optimization"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@0eb38997e5bc.(none)')


In [22]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

In [23]:
!git config --global user.name
!git config --global user.email

Amit Chandra Das
arickroy0@gmail.com


In [24]:
!git commit -m "Complete SERIAL 28 lightweight optimization"

[main 8c6819d] Complete SERIAL 28 lightweight optimization
 3 files changed, 8 insertions(+)
 create mode 100644 notebooks/28_lightweight_optimization.ipynb
 create mode 100644 results/lightweight_optimization_A8.csv
 create mode 100644 results/lstm_hidden_size_comparison_A8.csv


In [25]:
!git push origin main

Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 10.31 KiB | 405.00 KiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 3 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   e8e8b54..8c6819d  main -> main
